# 📘 Agentic Architectures 1：反思（Reflection）

欢迎来到“21 个关键 Agentic 架构”深度实践系列的第一个 Notebook。我们将从最基础也最强大的模式之一开始：**Reflection（反思）**。

这个模式可以把大语言模型（LLM）从“单次直出”的生成器，提升为更审慎、更稳健的推理器。反思型智能体不会只给出第一版答案，而是会后退一步，先批判、分析，再优化自己的结果。这种迭代式自我改进流程，是构建高可靠、高质量 AI 系统的核心基石之一。


### 定义
**Reflection** 架构指的是：智能体在给出最终答案前，先对自己的输出进行批评并修订。它不是一次性生成，而是执行“生成 -> 评估 -> 改进”的多步内部思考过程。这类似人类的写作流程：先起草，再审阅，再编辑，从而发现错误并提升质量。

### 高层工作流

1.  **生成（Generate）：** 智能体基于用户提示先产出初稿或初始解法。  
2.  **批评（Critique）：** 智能体切换为“评审者”角色，自问：*“这个答案可能哪里有问题？”*、*“缺了什么？”*、*“是否最优？”*、*“有逻辑漏洞或 bug 吗？”*。  
3.  **精修（Refine）：** 根据自我批评得到的洞察，生成改进后的最终版本。

### 适用场景 / 应用
*   **代码生成：** 初稿代码可能有 bug、效率低或缺少注释。Reflection 让智能体充当自己的代码评审，先修正再输出。  
*   **复杂摘要：** 面对高密度文档，一次总结常会漏细节。反思步骤可提升摘要的完整性和准确性。  
*   **创作与内容生产：** 邮件、博客、故事等初稿通常可优化。Reflection 有助于改进语气、清晰度与表达力度。

### 优势与局限
*   **优势：**
    *   **质量提升：** 能直接发现并纠正问题，输出更准确、更稳健、推理更完整。  
    *   **开销较低：** 概念简单，可用单个 LLM 实现，不依赖复杂外部系统。  
*   **局限：**
    *   **自我偏差：** 智能体仍受自身知识和偏见限制；它能修复“看得见”的问题，但无法凭空补齐“本来就不知道”的知识。  
    *   **时延与成本增加：** 至少需要两次 LLM 调用（生成 + 批评/修订），比单次生成更慢、更贵。


## 阶段 0：基础与环境准备

在构建反思型智能体之前，需要先准备运行环境，包括安装依赖库、导入模块，以及配置 API 密钥。


### 步骤 0.1：安装核心依赖

**我们要做什么：**  
安装本项目所需的核心 Python 库。`langchain-openai` 用于通过 OpenAI 兼容接口访问 DeepSeek 模型；`langchain` 与 `langgraph` 提供工作流编排能力；`python-dotenv` 管理 API 密钥；`rich` 用于更友好的终端输出展示。


In [2]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI  # 假设使用兼容 OpenAI 格式的 SDK

# 1. 加载 .env 文件中的变量到系统环境变量中
load_dotenv(override=True)

# 2. 从环境变量中提取 Key
# 使用 os.getenv 的好处是：如果变量不存在，它会返回 None 而不是报错
api_key = os.getenv("DEEPSEEK_API_KEY")

if not api_key:
    raise ValueError("未找到 API Key，请检查环境变量或 .env 文件！")


# 3. 初始化客户端并将 Key 注入
client = OpenAI(
    api_key=api_key, 
    base_url="https://api.deepseek.com/v1" # 以 DeepSeek 为例
)

# 4. 调用模型
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "你好"}]
)

print(response.choices[0].message.content)

你好！很高兴见到你！😊 我是DeepSeek，一个热心的AI助手，随时准备为你提供帮助。

无论你想聊天、寻求建议、解答问题，还是需要协助处理文档、分析信息，我都很乐意为你服务。我的知识截止到2024年7月，支持128K上下文，还可以处理你上传的各种文件（图像、文档、PDF等）。

今天有什么我可以帮助你的吗？不管是学习、工作还是生活中的问题，我都很愿意和你一起探讨！✨


### 步骤 0.2：导入库并配置密钥

**我们要做什么：**  
导入所有需要的组件；用 `python-dotenv` 从本地 `.env` 文件安全加载 DeepSeek API Key；并启用 LangSmith tracing，以便调试多步骤 Agent 工作流。

**需要你执行：** 在本 Notebook 同目录创建 `.env` 文件，并填写如下键值：
```
DEEPSEEK_API_KEY="your_deepseek_api_key_here"
LANGCHAIN_API_KEY="your_langsmith_api_key_here"
```


In [ ]:
import os
import json
from typing import List, TypedDict, Optional
from dotenv import load_dotenv

# DeepSeek and LangChain components
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field # Corrected import for Pydantic v2
from langgraph.graph import StateGraph, END

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown
from rich.syntax import Syntax

# --- API Key and Tracing Setup ---
load_dotenv()

# Set up LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Reflection (DeepSeek)"

# Check that the keys are set
if not os.environ.get("DEEPSEEK_API_KEY"):
    print("DEEPSEEK_API_KEY not found. Please create a .env file and set it.")
if not os.environ.get("LANGCHAIN_API_KEY"):
    print("LANGCHAIN_API_KEY not found. Please create a .env file and set it for tracing.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：构建 Reflection 的核心组件

一个可靠的 Reflection 架构不只是写个提示词。我们会把它设计为结构化的三段系统：**Generator（生成器）**、**Critic（批评器）**、**Refiner（精修器）**。为了保证稳定性，我们使用 Pydantic 模型定义每一步输出的结构约束。


### 步骤 1.1：用 Pydantic 定义数据模式

**我们要做什么：**  
定义 Pydantic 模型，作为 LLM 输出的“契约”。这能明确告知 LLM 每一步应输出什么结构，尤其适用于多步流程（上一步输出会成为下一步输入）。


In [3]:
class DraftCode(BaseModel):
    """Schema for the initial code draft generated by the agent."""
    code: str = Field(description="The Python code generated to solve the user's request.")
    explanation: str = Field(description="A brief explanation of how the code works.")

class Critique(BaseModel):
    """Schema for the self-critique of the generated code."""
    has_errors: bool = Field(description="Does the code have any potential bugs or logical errors?")
    is_efficient: bool = Field(description="Is the code written in an efficient and optimal way?")
    suggested_improvements: List[str] = Field(description="Specific, actionable suggestions for improving the code.")
    critique_summary: str = Field(description="A summary of the critique.")

class RefinedCode(BaseModel):
    """Schema for the final, refined code after incorporating the critique."""
    refined_code: str = Field(description="The final, improved Python code.")
    refinement_summary: str = Field(description="A summary of the changes made based on the critique.")

print("Pydantic models for Draft, Critique, and RefinedCode have been defined.")

Pydantic models for Draft, Critique, and RefinedCode have been defined.


**输出解读：**  
我们已经成功定义了数据结构。`Critique` 模型尤为关键：通过要求 `has_errors`、`is_efficient` 等明确字段，我们引导 LLM 做“结构化评估”，而不是笼统地“review code”，因此评审结果更可用。


### 步骤 1.2：初始化 DeepSeek LLM 与输出控制台

**我们要做什么：**  
初始化用于三个角色（Generator、Critic、Refiner）的 DeepSeek 语言模型。这里使用 `deepseek-chat` 作为默认模型，以兼顾推理质量与结构化输出兼容性；同时初始化 `rich` 控制台，用于美观输出。


In [26]:
# Use DeepSeek via the OpenAI-compatible API for generation and critique
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
    temperature=0.2,
)

# Initialize console for pretty printing
console = Console()

print("DeepSeek LLM and Console are initialized.")

DeepSeek LLM and Console are initialized.


### 步骤 1.3：创建 Generator 节点

**我们要做什么：**  
该节点只负责一件事：接收用户需求并生成第一版草稿。我们将 `DraftCode` Pydantic 模型绑定到 实例化的llm实例，确保输出结构正确。


In [5]:
def generator_node(state):
    """Generates the initial draft of the code."""
    console.print("--- 1. Generating Initial Draft ---")
    generator_llm = llm.with_structured_output(DraftCode)
    
    prompt = f"""You are an expert Python programmer. Write a Python function to solve the following request.
    Provide a simple, clear implementation and an explanation.
    
    Request: {state['user_request']}
    """
    
    draft = generator_llm.invoke(prompt)
    return {"draft": draft.model_dump()} # Corrected: use .model_dump()

### 步骤 1.4：创建 Critic 节点

**我们要做什么：**  
这是反思流程的核心。Critic 节点读取初稿，分析其中缺陷，并基于 `Critique` Pydantic 模型产出结构化评语。


In [6]:
def critic_node(state):
    """Critiques the generated code for errors and inefficiencies."""
    console.print("--- 2. Critiquing Draft ---")
    critic_llm = llm.with_structured_output(Critique)
    
    code_to_critique = state['draft']['code']
    
    prompt = f"""You are an expert code reviewer and senior Python developer. Your task is to perform a thorough critique of the following code.
    
    Analyze the code for:
    1.  **Bugs and Errors:** Are there any potential runtime errors, logical flaws, or edge cases that are not handled?
    2.  **Efficiency and Best Practices:** Is this the most efficient way to solve the problem? Does it follow standard Python conventions (PEP 8)?
    
    Provide a structured critique with specific, actionable suggestions.
    
    Code to Review:
    ```python
    {code_to_critique}
    ```
    """
    
    critique = critic_llm.invoke(prompt)
    return {"critique": critique.model_dump()} # Corrected: use .model_dump()

### 步骤 1.5：创建 Refiner 节点

**我们要做什么：**  
逻辑链路的最后一步是 Refiner。它同时接收“原始草稿 + 结构化批评”，并负责输出最终优化后的代码。


In [7]:
def refiner_node(state):
    """Refines the code based on the critique."""
    console.print("--- 3. Refining Code ---")
    refiner_llm = llm.with_structured_output(RefinedCode)
    
    draft_code = state['draft']['code']
    critique_suggestions = json.dumps(state['critique'], indent=2)
    
    prompt = f"""You are an expert Python programmer tasked with refining a piece of code based on a critique.
    
    Your goal is to rewrite the original code, implementing all the suggested improvements from the critique.
    
    **Original Code:**
    ```python
    {draft_code}
    ```
    
    **Critique and Suggestions:**
    {critique_suggestions}
    
    Please provide the final, refined code and a summary of the changes you made.
    """
    
    refined_code = refiner_llm.invoke(prompt)
    return {"refined_code": refined_code.model_dump()} # Corrected: use .model_dump()

**阶段 1 小结：**  
我们已经构建了反思型智能体的三个核心逻辑组件。每个组件都是职责单一、边界清晰的函数（节点）。每一步都使用结构化输出，保证数据能够稳定地从一个节点流向下一个节点。下一步，我们将用 LangGraph 对该流程进行编排。


## 阶段 2：使用 LangGraph 编排 Reflection 工作流


### 步骤 2.1：定义图状态（Graph State）

**我们要做什么：**  
“状态（state）”是图工作流的记忆中心。它会在节点间传递，各节点可读写状态。我们将使用 Python 的 `TypedDict` 定义 `ReflectionState`，统一保存流程中各阶段数据。


In [8]:
class ReflectionState(TypedDict):
    """Represents the state of our reflection graph."""
    user_request: str
    draft: Optional[dict]
    critique: Optional[dict]
    refined_code: Optional[dict]

print("ReflectionState TypedDict defined.")

ReflectionState TypedDict defined.


### 步骤 2.2：构建并可视化图结构

**我们要做什么：**  
使用 `StateGraph` 把节点组装成完整工作流。对于 Reflection 模式，流程是线性的：**Generate -> Critique -> Refine**。我们会先定义流程，再编译并可视化图，以确认结构符合预期。


In [9]:
graph_builder = StateGraph(ReflectionState)

# Add the nodes to the graph
graph_builder.add_node("generator", generator_node)
graph_builder.add_node("critic", critic_node)
graph_builder.add_node("refiner", refiner_node)

# Define the workflow edges
graph_builder.set_entry_point("generator")
graph_builder.add_edge("generator", "critic")
graph_builder.add_edge("critic", "refiner")
graph_builder.add_edge("refiner", END)

# Compile the graph
reflection_app = graph_builder.compile()

print("Reflection graph compiled successfully!")

# Visualize the graph
try:
    from IPython.display import Image, display
    png_image = reflection_app.get_graph().draw_png()
    display(Image(png_image))
except Exception as e:
    print(f"Graph visualization failed: {e}. Please ensure pygraphviz is installed.")

Reflection graph compiled successfully!
Graph visualization failed: Install pygraphviz to draw graphs: `pip install pygraphviz`.. Please ensure pygraphviz is installed.


**输出解读：**  
图已成功编译。可视化结果验证了我们预期的线性流程：状态从入口节点（`generator`）开始，经过 `critic` 与 `refiner`，最终到达 `__end__`。这个结构虽简洁，但已具备很强的可扩展能力，可直接用于执行。


## 阶段 3：端到端执行与评估

图编译完成后，就可以验证 Reflection 模式的实际效果。我们会给它一个“初稿大概率不够优”的编码任务，作为自我批评与修订的理想测试样例。


### 步骤 3.1：运行完整 Reflection 工作流

**我们要做什么：**  
调用编译后的 LangGraph 应用，请它实现“求第 n 个斐波那契数”的函数。我们会流式接收结果，并完整累积状态，便于最后检查全部中间步骤。


In [27]:
user_request = "Write a Python function to find the nth Fibonacci number."
initial_input = {"user_request": user_request}

console.print(f"[bold cyan]🚀 Kicking off Reflection workflow for request:[/bold cyan] '{user_request}'\n")

# Corrected: This loop correctly captures the final, fully-populated state
final_state = None
for state_update in reflection_app.stream(initial_input, stream_mode="values"):
    final_state = state_update

console.print("\n[bold green]✅ Reflection workflow complete![/bold green]")

🚀 Kicking off Reflection workflow for request: 'Write a Python function to find the nth Fibonacci number.'

--- 1. Generating Initial Draft ---

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


### 步骤 3.2：分析“前后对比”结果

**我们要做什么：**  
现在进入关键对比环节。我们将检查 `final_state` 中各阶段产物：初稿、批评意见、最终优化代码，从而直观看到 Reflection 带来的价值。


In [11]:
# Check if final_state is available and has the expected keys
if final_state and 'draft' in final_state and 'critique' in final_state and 'refined_code' in final_state:
    console.print(Markdown("--- ### Initial Draft ---"))
    console.print(Markdown(f"**Explanation:** {final_state['draft']['explanation']}"))
    # Use rich's Syntax for proper code highlighting
    console.print(Syntax(final_state['draft']['code'], "python", theme="monokai", line_numbers=True))

    console.print(Markdown("\n--- ### Critique ---"))
    console.print(Markdown(f"**Summary:** {final_state['critique']['critique_summary']}"))
    console.print(Markdown(f"**Improvements Suggested:**"))
    for improvement in final_state['critique']['suggested_improvements']:
        console.print(Markdown(f"- {improvement}"))

    console.print(Markdown("\n--- ### Final Refined Code ---"))
    console.print(Markdown(f"**Refinement Summary:** {final_state['refined_code']['refinement_summary']}"))
    console.print(Syntax(final_state['refined_code']['refined_code'], "python", theme="monokai", line_numbers=True))
else:
    console.print("[bold red]Error: The `final_state` is not available or is incomplete. Please check the execution of the previous cells.[/bold red]")

--- ### Initial Draft ---

Explanation: This function uses a recursive approach to calculate the nth Fibonacci number. It works by calling    
itself to calculate the (n-1)th and (n-2)th Fibonacci numbers and then adding them together. This approach is not  
efficient for large values of n due to the repeated calculations, but it is simple to understand and implement.

  1 def fibonacci(n):                                                                                              
  2     if n <= 0:                                                                                                 
  3         return 0                                                                                               
  4     elif n == 1:                                                                                               
  5         return 1                                                                                               
  6     else:                                                                                                      
  7         return fibonacci(n-1) + fibonacci(n-2)                                                                 
  8                                                                                                                

--- ### Critique ---

Summary: The function has potential bugs and inefficiencies. It should be revised to handle negative inputs and    
improve its time complexity.

...

 • The function does not follow PEP 8 conventions. The function name should be lowercase and the variable name     
   should be consistent.

 • Consider adding a docstring to describe the function's purpose and behavior.

--- ### Final Refined Code ---

Refinement Summary: The original code has been revised to handle negative inputs, improve its time complexity, and 
follow PEP 8 conventions. A docstring has been added to describe the function's purpose and behavior.

   1 def fibonacci(n):                                                                                             
   2     """Calculates the nth Fibonacci number.                                                                   
   3                                                                                                               
   4     Args:                                                                                                     
   5         n (int): The position of the Fibonacci number to calculate.                                           
   6                                                                                                               
   7     Returns:                                                                                                  
   8         int: The nth Fibonacci number.                                                                        
   9                                                                                                               
  10     Raises:                                                                                                   
  11         ValueError: If n is less than or equal to 0.                                                          
  12     """                                                                                                       
  13     if n < 0:                                                                                                 
  14         raise ValueError("n must be a non-negative integer")                                                  
  15     elif n == 0:                                                                                              
  16         return 0                                                                                              
  17     elif n == 1:                                                                                              
  18         return 1                                                                                              
  19     else:                                                                                                     
  20         fib = [0, 1]                                                                                          
  21         for i in range(2, n + 1):                                                                             
  22             fib.append(fib[i-1] + fib[i-2])                                                                   
  23         return fib[n]                                                                                         
  24                                                                                                               

**输出解读：**  
结果非常清晰地展示了 Reflection 的价值。

1.  **初始草稿（Initial Draft）** 很可能给出简单递归实现。虽然逻辑正确，但会重复计算子问题，时间复杂度呈指数增长，效率很差。  
2.  **批评结果（Critique）** 准确指出了这一核心问题，并建议改为迭代写法以避免重复计算。  
3.  **最终精修代码（Final Refined Code）** 成功落实了批评意见：从慢速递归改为高效迭代，通过循环与两个变量追踪序列。

这不是“修修语法”级别的小改动，而是算法层面的实质优化。Reflection 的价值正在于此：智能体能够通过自我评审，得到更稳健、更可扩展的解法。


### 步骤 3.3：量化评估（LLM-as-a-Judge）

**我们要做什么：**  
为了让分析更客观，我们引入另一个 LLM 作为“裁判”，对初稿与最终代码进行打分对比，从量化角度衡量 Reflection 带来的提升。


In [12]:
class CodeEvaluation(BaseModel):
    """Schema for evaluating a piece of code."""
    correctness_score: int = Field(description="Score from 1-10 on whether the code is logically correct.")
    efficiency_score: int = Field(description="Score from 1-10 on the code's algorithmic efficiency.")
    style_score: int = Field(description="Score from 1-10 on code style and readability (PEP 8). ")
    justification: str = Field(description="A brief justification for the scores.")

judge_llm = llm.with_structured_output(CodeEvaluation)

def evaluate_code(code_to_evaluate: str):
    prompt = f"""You are an expert judge of Python code. Evaluate the following function on a scale of 1-10 for correctness, efficiency, and style. Provide a brief justification.
    
    Code:
    ```python
    {code_to_evaluate}
    ```
    """
    return judge_llm.invoke(prompt)

if final_state and 'draft' in final_state and 'refined_code' in final_state:
    console.print("--- Evaluating Initial Draft ---")
    initial_draft_evaluation = evaluate_code(final_state['draft']['code'])
    console.print(initial_draft_evaluation.model_dump()) # Corrected: use .model_dump()

    console.print("\n--- Evaluating Refined Code ---")
    refined_code_evaluation = evaluate_code(final_state['refined_code']['refined_code'])
    console.print(refined_code_evaluation.model_dump()) # Corrected: use .model_dump()
else:
    console.print("[bold red]Error: Cannot perform evaluation because the `final_state` is incomplete.[/bold red]")

--- Evaluating Initial Draft ---

{
    'correctness_score': 2,
    'efficiency_score': 2,
    'style_score': 2,
    'justification': 'The function has a time complexity of O(2^n) due to the repeated computation of the same 
subproblems, which makes it inefficient. The code also does not follow the PEP 8 style guide as it does not have a 
consistent indentation and the function name does not follow the conventional snake_case naming convention. The 
base cases are also not handled correctly, as the function returns 0 for n <= 0, but the Fibonacci sequence is 
typically defined as 0, 1, 1, 2, 3, 5, 8, 13, ...'
}

--- Evaluating Refined Code ---

{
    'correctness_score': 8,
    'efficiency_score': 6,
    'style_score': 9,
    'justification': 'The code is correct for calculating the nth Fibonacci number. However, it has a time 
complexity of O(n) due to the use of a loop, which is not efficient for large values of n. The style is good, 
following PEP 8 conventions, but it could be improved by using a more efficient algorithm like memoization or 
dynamic programming.'
}

**输出解读：**  
`LLM-as-a-Judge` 的评分结果给出了量化证据：初稿通常在“正确性”上不差，但“效率”得分较低；而优化后代码在正确性与效率上都明显更高。  
这说明 Reflection 不只是“改了代码”，而是以可度量的方式**提升了质量**。


## 结论

在本 Notebook 中，我们基于 Nebius AI Studio 模型，完整地实现、运行并评估了 **Reflection** 架构的端到端智能体流程。实践表明，这个简单但强大的模式，能够把基础型 LLM 生成器升级为更可靠的复杂问题求解器。

通过将流程拆分为 **Generate**、**Critique**、**Refine** 三步，并借助 LangGraph 完成编排，我们构建了一个可稳定识别并修复关键缺陷的系统。从低效递归到高效迭代的改进，证明了 Reflection 是迈向高质量 Agent 系统的重要基础技术。
